### Load source
**Process:** Use a GPU runtime with no Google Drive mounted. Upload remaining-projects-source.zip.

**Expected output:** Extracted source and command logging.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,time
WORK=Path("/content");os.chdir(WORK)
archive=WORK/"remaining-projects-source.zip"
if not archive.exists():
    from google.colab import files
    files.upload()
if not archive.exists():raise RuntimeError("Upload remaining-projects-source.zip")
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (WORK/name).resolve().is_relative_to(WORK):raise ValueError("Unsafe archive path")
    z.extractall(WORK)
ROOT=WORK/"remaining-projects"
LOGS=ROOT/"verification";LOGS.mkdir(exist_ok=True)
def run(args,cwd=None,env=None):
    logfile=LOGS/(str(time.time_ns())+".log")
    print("Running:",args,"Log:",logfile,flush=True)
    with logfile.open("w") as log:
        with subprocess.Popen(args,cwd=cwd,env={**os.environ,**(env or {}),"PYTHONUNBUFFERED":"1"},
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True);log.write(line);log.flush()
            code=p.wait()
    if code:raise RuntimeError(f"Exit {code}; see {logfile}")


### Prepare local model
**Process:** Install editor/model dependencies, detect CUDA and download a pinned Qwen3-8B snapshot.

**Expected output:** GPU details, model download progress and saved model revision.


In [ ]:
run([sys.executable,"-m","pip","install","-r",str(ROOT/"autonomous-code-editor/requirements.txt"),
    "transformers>=5,<6","torch>=2.5,<3","accelerate>=1,<2","bitsandbytes>=0.45,<1"])
import torch
if not torch.cuda.is_available():raise RuntimeError("Select a CUDA GPU runtime")
print(torch.cuda.get_device_name(0),flush=True)
from huggingface_hub import HfApi,snapshot_download
revision=HfApi().model_info("Qwen/Qwen3-8B").sha
MODEL=snapshot_download("Qwen/Qwen3-8B",revision=revision,ignore_patterns=["*.gguf","*.msgpack","*.h5"])
(LOGS/"model.json").write_text(json.dumps({"id":"Qwen/Qwen3-8B","revision":revision}))


### Run actual model repair probes
**Process:** Generate replacement functions for three disclosed tasks and verify candidates in disposable Git worktrees, with at most three attempts per task. Generated code executes in this runtime; do not mount personal Drive or add credentials.

**Expected output:** A report with actual proposals, failures, verified diffs and attempt counts. Three synthetic probes cannot establish a general repair-success percentage.


In [ ]:
run([sys.executable,"-m","benchmarks.model_repair","--model",MODEL,"--allow-local-execution",
    "--output",str(LOGS/"repair-model-probes.json")],cwd=ROOT/"autonomous-code-editor")
print((LOGS/"repair-model-probes.json").read_text())


### Download repair evidence
**Process:** Archive the model revision, logs and per-attempt results.

**Expected output:** editor-model-results.zip download.


In [ ]:
import shutil
archive=shutil.make_archive(str(WORK/"editor-model-results"),"zip",ROOT,"verification")
from google.colab import files
files.download(archive)
